In [2]:
# Cell 1: Imports and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import pyedflib
import os
import random
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

np.random.seed(1)
random.seed(1)

# base data directory
data_dir = '../data/'

print("Imports got loaded")

Imports got loaded


In [3]:
summary_path = '../data/chb12/chb12-summary.txt'
print(f"File exists: {os.path.exists(summary_path)}")
print(f"File size: {os.path.getsize(summary_path)} bytes")

with open(summary_path, 'r', encoding='utf-8', errors='ignore') as f:
    content = f.read()
print(f"Content length: {len(content)}")
print("\nFirst 1000 characters:")
print(content[:1000])

File exists: True
File size: 7384 bytes
Content length: 7384

First 1000 characters:
Data Sampling Rate: 256 Hz
*************************

Channels in EDF Files:
**********************
Channel 1: FP1-F7
Channel 2: F7-T7
Channel 3: T7-P7
Channel 4: P7-O1
Channel 5: -
Channel 6: FP1-F3
Channel 7: F3-C3
Channel 8: C3-P3
Channel 9: P3-O1
Channel 10: -
Channel 11: FZ-CZ
Channel 12: CZ-PZ
Channel 13: -
Channel 14: FP2-F4
Channel 15: F4-C4
Channel 16: C4-P4
Channel 17: P4-O2
Channel 18: -
Channel 19: FP2-F8
Channel 20: F8-T8
Channel 21: T8-P8
Channel 22: P8-O2
Channel 23: -
Channel 24: P7-T7
Channel 25: T7-FT9
Channel 26: FT9-FT10
Channel 27: FT10-T8
Channel 28: T8-P8

File Name: chb12_06.edf
File Start Time: 22:44:34
File End Time: 23:44:40
Number of Seizures in File: 2
Seizure 1 Start Time: 1665 seconds
Seizure 1 End Time: 1726 seconds
Seizure 2 Start Time: 3415 seconds
Seizure 2 End Time: 3447 seconds

File Name: chb12_08.edf
File Start Time: 00:46:38
File End Time: 1:46:38
Number of Seizu

In [4]:
# Cell 2: Parse all 22 patient summary files — robust version

def parse_summary_file(patient_id):
    """
    Parse CHB-MIT summary file to extract seizure timings for one patient.
    Handles multiple summary file formats across different patients.
    
    Returns list of dicts:
    [{'file': 'chb01_03.edf', 'start': 2996, 'end': 3036, 'patient': 'chb01'}, ...]
    """
    summary_path = os.path.join(data_dir, patient_id, f'{patient_id}-summary.txt')

    # check if the summary file path exists or not
    if not os.path.exists(summary_path):
        print(f"   No summary file for {patient_id}")
        return []
    
    with open(summary_path, 'r') as f:
        content = f.read()
    
    seizures = []
    current_file = None # track which file we are reading
    
    lines = content.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        
        # get current file name
        # summary files list seizures under each file they appear in
        if 'File Name:' in line:
            current_file = line.split('File Name:')[-1].strip()
        
        # handle both "Seizure Start Time" and "Seizure N Start Time"
        # look for seizure start time lines
        if current_file and ('Seizure' in line) and ('Start Time' in line):
            # extract seconds: grab all digits before 'seconds'
            try:
                # extract number of seconds from the line
                # e.g. "Seizure 1 Start Time: 2996 seconds" -> 2996
                start_part = line.split('Start Time:')[-1].strip()
                start = int(''.join(filter(str.isdigit, start_part.split('seconds')[0])))
                
                # look for end time in next few lines
                # store the seizure end time in end
                for j in range(i+1, min(i+5, len(lines))):
                    next_line = lines[j].strip()
                    if 'End Time' in next_line:
                        end_part = next_line.split('End Time:')[-1].strip()
                        end = int(''.join(filter(str.isdigit, end_part.split('seconds')[0])))
                        # store seizure info as a dict
                        seizures.append({
                            'file':    current_file, # which EDF file the seizure is in
                            'start':   start, # seizure onset in seconds
                            'end':     end, # seizure end in seconds
                            'patient': patient_id # which patient
                        })
                        break
            except:
                pass
        
        i += 1
    
    return seizures

# Parse all 23 patients
# skip chb21, since this is the same patient a year later for some reason
patient_ids = [f'chb{str(i).zfill(2)}' for i in range(1, 25) if i != 21]

all_patients_seizures = {}
total_seizures = 0

print("Parsing summary files \n")
for patient_id in patient_ids:
    seizures = parse_summary_file(patient_id)
    if len(seizures) > 0:
        # store seizures for this patient
        all_patients_seizures[patient_id] = seizures
        total_seizures += len(seizures)
        print(f" {patient_id}: {len(seizures)} seizures")
    else:
        print(f" {patient_id}: no seizures found — check summary file format")

print(f"\nTotal patients with seizures: {len(all_patients_seizures)}")
print(f"Total seizures across all patients: {total_seizures}")

Parsing summary files 

 chb01: 7 seizures
 chb02: 3 seizures
 chb03: 7 seizures
 chb04: 4 seizures
 chb05: 5 seizures
 chb06: 10 seizures
 chb07: 3 seizures
 chb08: 5 seizures
 chb09: 4 seizures
 chb10: 7 seizures
 chb11: 3 seizures
 chb12: 40 seizures
 chb13: 12 seizures
 chb14: 8 seizures
 chb15: 20 seizures
 chb16: 10 seizures
 chb17: 3 seizures
 chb18: 6 seizures
 chb19: 3 seizures
 chb20: 8 seizures
 chb22: 3 seizures
 chb23: 7 seizures
 chb24: 16 seizures

Total patients with seizures: 23
Total seizures across all patients: 194


In [5]:
# Cell 3: Feature extraction functions — 69 features (3 × 23 channels)

sfreq = 256

# standard channel names used across all CHB-MIT patients
standard_channels = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ',  'CZ-PZ', 'P7-T7', 'T7-FT9',
    'FT9-FT10', 'FT10-T8', 'T8-P8'
]

# Read summary file  to get valid channel indices and names for a patient
# Only reads the first channel since some summary files later on change the name of the channel (irrelvenat)
# Skip channels labeled "-" that contain no real data

def get_patient_channels(patient_id):
    summary_path = os.path.join(data_dir, patient_id, f'{patient_id}-summary.txt')
    valid_channels = []
    found_channels = False
    with open(summary_path, 'r', errors='ignore') as f:
        lines = f.readlines()
    for line in lines:
        line = line.strip()
        # this line marks the start of the channel list in the summary file
        if 'Channels in EDF' in line:
            found_channels = True
            continue

        # once hit the first file entry stop reading channels
        # prevents from reading channel updates
        # stop here as not read channel changes listed later
        if found_channels and line.startswith('File Name:'):
            break

        # read lines that look like "Channel 1: FP-FP7"
        # grab each channel line number and name
        if found_channels and line.startswith('Channel ') and ':' in line:
            parts = line.split(':')
            if len(parts) >= 2:
                try:
                    ch_name = parts[1].strip()
                    ch_idx = int(parts[0].replace('Channel', '').strip()) - 1
                    # skip EEG channels with no name: theyr pretty useless
                    if ch_name != '-' and ch_name != '':
                        valid_channels.append((ch_idx, ch_name))
                except ValueError:
                    continue
    # returns a tuple of all the valid channels for a patient
    return valid_channels


# Opens an EDF file and reads one channel of raw EEG voltage data
# EDF is the standard file format of the CHB mit dataset
# reutnrs an array of voltage values
def extract_eeg_from_file(patient_id, filename, channel_idx):
    filepath = os.path.join(data_dir, patient_id, filename)
    f = pyedflib.EdfReader(filepath)
    # skip if theres a channel missing
    if channel_idx >= f.signals_in_file:
        f.close()
        return None
    sig = f.readSignal(channel_idx)
    f.close()
    return sig

def calculate_entropy(seg):
    # count how many samples fall in each of 50 voltage bins
    hist, _ = np.histogram(seg, bins=50)
    # convert counts to probabilities (must sum to 1)
    hist = hist / np.sum(hist)
    # remove empty bins to avoid log(0) which is undefined
    hist = hist[hist > 0]
    # Shannon entropy formula
    return -np.sum(hist * np.log2(hist))

def calculate_line_length(seg):
    # difference between each sample and the next one
    differences = np.diff(seg)
    # take absolute value
    absolute_differences = np.abs(differences)
    # sum everything up to get total signal variation
    return np.sum(absolute_differences)

def calculate_high_freq_power(seg, sfreq=256):
    # nperseg is the chunk size for Welch's method
    # cap at 1024 samples to keep computation fast
    nperseg = min(1024, len(seg))
    # f = frequency array, pxx = power at each frequency
    f, pxx = signal.welch(seg, sfreq, nperseg=nperseg)
    # add up all power at 20 Hz and above
    return np.sum(pxx[f >= 20])

def extract_features_from_window(eeg_channels, channel_info, start_sec, end_sec, sfreq):
    """
    This is the main feature extraction function.
    
    For one 2-minute window of EEG it:
    1. Loops through all 23 standard channels
    2. Extracts entropy, line length, and high-freq power from each channel
    3. Returns one array of 69 numbers (3 features × 23 channels)
    
    If a channel is missing for this patient it fills with 0
    so the output is always exactly 69 numbers regardless of patient.
    
    This 69-number array becomes one row in the X matrix.
    """
    # build a dictionary so we can look up channels by name
    # maps channel_name -> raw EEG signal array
    channel_lookup = {}
    for i, (ch_idx, ch_name) in enumerate(channel_info):
        if i < len(eeg_channels) and eeg_channels[i] is not None:
            channel_lookup[ch_name] = eeg_channels[i]

    # convert window start/end from seconds to sample numbers
    # at 256 Hz, 1 second = 256 samples, 2 minutes = 30,720 samples
    start_idx = int(start_sec * sfreq)
    end_idx   = int(end_sec   * sfreq)

    feature_vector = [] # will hold 69 numbers when done
    feature_names  = [] # will hold names like FP-F7: entropy


    # loop through all 23 channels in fixed order
    for ch_name in standard_channels:
        if ch_name in channel_lookup:
            sig = channel_lookup[ch_name]
            # slice out just the 2-minute segment 
            seg = sig[start_idx:end_idx]
            if len(seg) > 0:
                # calculate all 3 features for this channel
                feature_vector.extend([
                    calculate_entropy(seg),
                    calculate_line_length(seg),
                    calculate_high_freq_power(seg, sfreq),
                ])
            else:
                # segnment is empty
                feature_vector.extend([0, 0, 0])
        else:
            # channel doesn't exist for the patient
            feature_vector.extend([0, 0, 0])

        # store the name of each feature
        feature_names.extend([
            f'{ch_name}_entropy',
            f'{ch_name}_line_length',
            f'{ch_name}_hf_power',
        ])

    # return numpy array
    return np.array(feature_vector), feature_names

def get_previous_file(patient_id, current_file):
    """
    This function finds the file that came right before the current one.
    
    Example: chb01_03.edf -> returns chb01_02.edf
    Returns None if there is no previous file.
    """
    patient_dir = os.path.join(data_dir, patient_id)
    all_files = sorted([f for f in os.listdir(patient_dir) if f.endswith('.edf')])
    if current_file not in all_files:
        return None
    idx = all_files.index(current_file)
    return all_files[idx-1] if idx > 0 else None

def load_stitched_preictal(patient_id, sz_file, sz_start, preictal_sec, sfreq, channel_info):
    """
    Loads the full preictal zone (20 minutes) before a seizure.
    
    Stitch the previous file onto the beginning.
    Take the last (1200 - 327 = 873) seconds from the previous file
    and join it with the 327 seconds from the current file.
    This gives us a continuous 1200-second signal ending at seizure onset.
    
    CHB-MIT files are sequential recordings with
    gaps of less than 10 seconds between files.
    
    Returns the stitched EEG data and the time range to extract windows from.
    """
    need_from_prev = preictal_sec - sz_start
    eeg_channels = []
    for ch_idx, ch_name in channel_info:
        try:
            # step 1: load the seizure file
            eeg_current = extract_eeg_from_file(patient_id, sz_file, ch_idx)
            if eeg_current is None:
                eeg_channels.append(None)
                continue

            # step 2: cut off everything at and after the seizure onset
            eeg_current = eeg_current[:int(sz_start * sfreq)]
            if need_from_prev > 0:
                # step 3: need more data so load the previous file
                prev_file = get_previous_file(patient_id, sz_file)
                if prev_file:
                    eeg_prev = extract_eeg_from_file(patient_id, prev_file, ch_idx)
                    if eeg_prev is not None:
                        eeg_channels.append(np.concatenate([
                            eeg_prev[-int(need_from_prev*sfreq):],
                            eeg_current
                        ]))
                    else:
                        # previous file failed to load
                        eeg_channels.append(eeg_current)
                else:
                    # no previous file exists
                    eeg_channels.append(eeg_current)
            else:
                # no stiching needed
                eeg_channels.append(eeg_current)
        except:
            eeg_channels.append(None)

    # return the stitched channels and where to extract windows from
    if need_from_prev > 0:
        return eeg_channels, 0, preictal_sec
    else:
        return eeg_channels, sz_start - preictal_sec, sz_start

print(f"  Features per window: 3 × 23 = {3*23}")

  Features per window: 3 × 23 = 69


In [6]:
# Cell 4: Build labeled dataset for all 23 patients — with 1-hour gap rule

WINDOW_SEC   = 120   # each window is 2 minutes of EEG
STEP_SEC     = 120   # non-overlapping — next window starts right after previous ends
PREICTAL_SEC = 1200  # 20 minutes before seizure = preictal zone

def is_file_safe(patient_id, clean_file, seizure_files_set, all_files, min_gap_files=1):
    """
    Checks if a clean file is far enough away from any seizure file.
    
    Files immediately before a seizure might contain preictal brain activity.
    Files immediately after a seizure might contain postictal brain activity.
    Both would corrupt our interictal label if included.
    
    Since each CHB-MIT file is roughly 1 hour long, requiring 1 file gap
    between a clean file and any seizure file gives us roughly 1 hour of separation.
    
    Returns True if the file is safe to use for interictal sampling.
    """
    if clean_file not in all_files:
        return False
    
    clean_idx = all_files.index(clean_file)
    
    for sz_file in seizure_files_set:
        if sz_file not in all_files:
            continue
        sz_idx = all_files.index(sz_file)
        # if the clean file is within 1 position of a seizure file then exclude it
        if abs(clean_idx - sz_idx) <= min_gap_files:
            return False
    
    return True  # file is at least 2 positions away from all seizure files

# these three lists build up as loop through all patients
# X holds feature vectors, y holds labels, patient_labels tracks which patient each window came from
X = []
y = []
patient_labels = []
feature_names_list = None  # stores feature names from first window which are the same for all windows

print("Building dataset for all 23 patients with 1-hour gap rule...\n")

# loop through every patient and their seizures
for patient_id, seizures in all_patients_seizures.items():
    print(f"\nProcessing {patient_id} ({len(seizures)} seizures)")

    patient_dir   = os.path.join(data_dir, patient_id)
    channel_info  = get_patient_channels(patient_id)  # get this patient's channel layout
    
    # get all EDF files for this patient in chronological order
    all_edf       = sorted([f for f in os.listdir(patient_dir) if f.endswith('.edf')])
    
    # identify which files contain seizures which are then excluded from interictal sampling
    seizure_files = set([sz['file'] for sz in seizures])

    # filter to files that are safe for interictal sampling
    # must be seizure-free and at least 1 file away from any seizure file
    clean_files_pt = [f for f in all_edf
                      if f not in seizure_files
                      and is_file_safe(patient_id, f, seizure_files, all_edf, min_gap_files=1)]

    if len(clean_files_pt) == 0:
        print(f"  ✗ No safe clean files — skipping {patient_id}")
        continue

    preictal_count   = 0
    interictal_count = 0

    # PREICTAL WINDOWS
    # for each seizure extract the 20 minutes before onset
    # these windows represent brain activity approaching a seizure -> label = 1
    for sz in seizures:
        sz_start = sz['start']  # seizure onset time in seconds within the file
        sz_file  = sz['file']   # which EDF file this seizure is in

        # load the preictal EEG — stitches previous file if seizure is too early in recording
        # returns the EEG data + the time range to extract windows from
        eeg_channels, extract_start, extract_end = load_stitched_preictal(
            patient_id, sz_file, sz_start, PREICTAL_SEC, sfreq, channel_info
        )

        # skip this seizure if we dont have enough data for even one window
        if extract_end - extract_start < WINDOW_SEC:
            continue

        # slide a 2-minute window across the full 20-minute preictal zone
        # with no overlap
        current_time = extract_start
        while current_time + WINDOW_SEC <= extract_end:
            
            # extract 69 features from this 2-minute window across all 23 channels
            fv, fn = extract_features_from_window(
                eeg_channels, channel_info,
                current_time, current_time + WINDOW_SEC, sfreq
            )
            
            X.append(fv)                      # add 69-element feature vector to dataset
            y.append(1)                        # label = 1 (preictal)
            patient_labels.append(patient_id)  # track which patient this window came from

            # save feature names from first window
            if feature_names_list is None:
                feature_names_list = fn

            current_time += STEP_SEC  # move to next non-overlapping window
            preictal_count += 1

    # INTERICTAL WINDOWS
    # extract windows from clean seizure-free files
    # these represent normal brain activity -> label = 0
    random.seed(1)  # fixed seed so same windows are sampled every run

    # build a list of every possible 2-minute window across all safe clean files
    # each entry is (filename, start_time_in_seconds)
    possible = []
    for cf in clean_files_pt:
        try:
            fp  = os.path.join(patient_dir, cf)
            f   = pyedflib.EdfReader(fp)
            dur = f.getNSamples()[0] / sfreq  # file duration in seconds
            f.close()
            
            # latest a window can start so it doesnt run past end of file
            max_start = dur - WINDOW_SEC
            
            # every possible non-overlapping 2-minute start time in this file
            starts = list(range(0, int(max_start), WINDOW_SEC))
            for s in starts:
                possible.append((cf, s))
        except:
            continue

    # randomly sample exactly as many interictal windows as preictal windows
    # this keeps the dataset balanced 50/50 for this patient
    n_to_sample = min(preictal_count, len(possible))
    if n_to_sample == 0:
        print(f" No interictal windows available, skipping")
        continue

    sampled = random.sample(possible, n_to_sample)

    # extract features from each sampled interictal window
    # cache loaded files to avoid reloading the same file multiple times
    loaded_files = {}
    for cf, start in sampled:
        
        # load this file's channels if not already in cache
        if cf not in loaded_files:
            eeg_ch = []
            for ch_idx, ch_name in channel_info:
                try:
                    eeg_ch.append(extract_eeg_from_file(patient_id, cf, ch_idx))
                except:
                    eeg_ch.append(None)
            loaded_files[cf] = eeg_ch

        # extract 69 features from this 2-minute interictal window
        fv, fn = extract_features_from_window(
            loaded_files[cf], channel_info,
            start, start + WINDOW_SEC, sfreq
        )
        
        X.append(fv)                      # add feature vector to dataset
        y.append(0)                        # label = 0 
        patient_labels.append(patient_id)  # track which patient
        interictal_count += 1

    
    loaded_files = {}

    print(f"  {preictal_count} preictal + {interictal_count} interictal windows")

# convert python lists to numpy arrays for sklearn
# X shape: (total_windows, 69): each row is one 2-minute window
# y shape: (total_windows,): each value is 0 or 1
# patient_labels shape: (total_windows,): which patient each window belongs to
X = np.array(X)
y = np.array(y)
patient_labels = np.array(patient_labels)

print(f"  Total windows:      {len(y)}")
print(f"  Preictal (y=1):     {np.sum(y == 1)}")
print(f"  Interictal (y=0):   {np.sum(y == 0)}")
print(f"  X shape:            {X.shape}")
print(f"  Patients:           {len(np.unique(patient_labels))}")
print(f"  Class ratio:        {np.sum(y==1)/len(y):.2f}")

Building dataset for all 23 patients with 1-hour gap rule...


Processing chb01 (7 seizures)
  70 preictal + 70 interictal windows

Processing chb02 (3 seizures)
  30 preictal + 30 interictal windows

Processing chb03 (7 seizures)
  70 preictal + 70 interictal windows

Processing chb04 (4 seizures)
  40 preictal + 40 interictal windows

Processing chb05 (5 seizures)
  50 preictal + 50 interictal windows

Processing chb06 (10 seizures)
  100 preictal + 100 interictal windows

Processing chb07 (3 seizures)
  30 preictal + 30 interictal windows

Processing chb08 (5 seizures)
  50 preictal + 50 interictal windows

Processing chb09 (4 seizures)
  40 preictal + 40 interictal windows

Processing chb10 (7 seizures)
  70 preictal + 70 interictal windows

Processing chb11 (3 seizures)
  30 preictal + 30 interictal windows

Processing chb12 (40 seizures)
  400 preictal + 58 interictal windows

Processing chb13 (12 seizures)
  120 preictal + 120 interictal windows

Processing chb14 (8 seizures)
  

In [8]:
# Cell 5: Train Random Forest with LOPOCV across all 23 patients

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Step 1: per-patient z-score first
X_normalized = X.copy().astype(float)
for pid in np.unique(patient_labels):
    mask = patient_labels == pid
    X_normalized[mask] = StandardScaler().fit_transform(X_normalized[mask])

# Step 2: global scaling on top
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_normalized)
print(f"  X shape: {X_normalized.shape}")
print(f"  y shape: {y.shape}")
print(f"  Patients: {len(np.unique(patient_labels))}")

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',  
    random_state=1,
    n_jobs=-1
)

# LOPOCV leave one patient out at a time, all 23 patients
logo = LeaveOneGroupOut()

y_pred = cross_val_predict(
    rf, X_scaled, y,
    groups=patient_labels,
    cv=logo
)


# results
print("\n" + "="*55)
print("LOPOCV RESULTS: ALL 23 PATIENTS")
print("="*55)
print(classification_report(y, y_pred, target_names=['Interictal', 'Preictal']))

cm = confusion_matrix(y, y_pred)
print("Confusion Matrix:")
print(f"                   Predicted")
print(f"                   Interictal  Preictal")
print(f"Actual Interictal      {cm[0][0]}          {cm[0][1]}")
print(f"Actual Preictal        {cm[1][0]}          {cm[1][1]}")

sensitivity = cm[1][1] / (cm[1][1] + cm[1][0])
specificity = cm[0][0] / (cm[0][0] + cm[0][1])

print(f"\nSensitivity: {sensitivity:.2%}")
print(f"Specificity: {specificity:.2%}")

try:
    auc = roc_auc_score(y, y_pred)
    print(f"AUC:         {auc:.3f}")
except:
    print("AUC: could not compute")

  X shape: (3407, 69)
  y shape: (3407,)
  Patients: 23

LOPOCV RESULTS: ALL 23 PATIENTS
              precision    recall  f1-score   support

  Interictal       0.46      0.41      0.43      1467
    Preictal       0.59      0.64      0.61      1940

    accuracy                           0.54      3407
   macro avg       0.52      0.52      0.52      3407
weighted avg       0.53      0.54      0.53      3407

Confusion Matrix:
                   Predicted
                   Interictal  Preictal
Actual Interictal      599          868
Actual Preictal        708          1232

Sensitivity: 63.51%
Specificity: 40.83%
AUC:         0.522
